In [5]:

from science_jubilee.Machine import Machine, TipRackManager
from science_jubilee.tools.Tool import Tool
import logging
import math
from pycont.controller import PumpIO
import pycont.controller as ctrl
import gc, sys, importlib


##########################################
#       TriContinent C3000 Set Up        #
##########################################


# Try close old objects if they exist
for name in ("pump", "controller"):
    obj = globals().get(name)
    if obj is None:
        continue
    try:
        # Best case: your MultiPumpController gets a close() (see patch below)
        if hasattr(obj, "close"):
            obj.close()
        # Fallbacks:
        elif hasattr(obj, "_io") and hasattr(obj._io, "close"):
            obj._io.close()
        elif hasattr(obj, "pumps"):
            # Close underlying PumpIO(s) through any pump we can reach
            for p in getattr(obj, "pumps", {}).values():
                try:
                    if hasattr(p, "_io") and hasattr(p._io, "close"):
                        p._io.close()
                except Exception:
                    pass
    except Exception:
        pass
    finally:
        del globals()[name]
gc.collect()
# Hard-reload the module so your code edits take effect
importlib.reload(ctrl)
# Recreate fresh objects

# Remove the EEPROM flash line - only run this once!
SETUP_CONFIG_FILE = r"C:/Users/Blab2/Desktop/Project/pycont/pump_setup_config.json"
controller = ctrl.MultiPumpController.from_configfile(SETUP_CONFIG_FILE)
pump = controller.pumps["water"]

# Use smart_initialize instead
pump.smart_initialize()


#=============================================================================================================


##########################################
#             Jubilee Set Up             #
##########################################

m = Machine(address="192.168.1.2")
m.load_tool(Tool(name="electrode", index=0))
m.load_tool(Tool(name="dummy_probe", index=1))
m.load_tool(Tool(name="tube", index=2))

deck = m.load_deck("lab_automation_deck_B-Lab")  # Replace with your deck file name
labware1 = deck.load_labware("nest_96_wellplate_200ul", 2) 
labware2 = deck.load_labware("blab_12_wellplate_20ul", 3) 
labware3 = deck.load_labware("blab_4_wellplate_50ul", 0) 
labware4 = deck.load_labware("blab_2_wellplate_100ul", 4)
tiprack = deck.load_labware("opentrons_96_tiprack_300ul", 1)
tip_manager = TipRackManager(tiprack, m)

m.tool_lock()
m.tool_unlock()


0
response in connect:  [False, False, False, True]


TypeError: the JSON object must be str, bytes or bytearray, not NoneType

In [ ]:
for well1 in labware1:                       
    well1.apply_offset((-60.9, -40.8, 0))       #Offsets for NEST wellplate! 

for well2 in labware2:                        
    well2.apply_offset((-58.7, -42, 0))       #Offsets for B-Lab wellplate of 20mL! 

for well3 in labware3:                        
    well3.apply_offset((-66.6, -42, 0))       #Offsets for B-Lab wellplate of 50mL! 

for well4 in labware4:                        
    well4.apply_offset((-66, -42, 0))         #Offsets for B-Lab wellplate of 100mL! 

for tips in tiprack:                        
    tips.apply_offset((-59.8, -44, 0))          #Offsets for piette tip rack! 

In [ ]:
import sys
import threading

sys.path.append(r"C:\Users\Blab2\Desktop\Project")
from spectrum_analyzer import start_monitoring

folder = r"C:\Users\Blab2\Desktop\Project\oceanview_data"

# Run monitoring in background
thread = threading.Thread(target=start_monitoring, args=(folder,), daemon=True)
thread.start()

print("Monitoring started in background.")

Monitoring started in background.

🎨 DYE ANALYSIS SYSTEM - MONITORING ACTIVE 🎨
📁 Watching folder: C:\Users\Blab2\Desktop\Project\oceanview_data
🔬 Monitoring for: Yellow 5, Yellow 6, Red 40, Blue 1
⏸️  Press Ctrl+C to stop monitoring...



In [ ]:
tip = tiprack["A1"]
m.move_to(x=tip.x, y=tip.y, s=2000)


MachineStateError: Error: machine must first be homed.

In [ ]:
m.pickup_tool(1)    


🔍 NEW FILE DETECTED: QEP075821__0__12-05-48-380.txt

TRANSMISSION SPECTRUM ANALYSIS
File: QEP075821__0__12-05-48-380.txt
Transmission Rise Point: 608.93 nm
Intensity at inflection: 18636.38
Rate of rise: 828.23
Identified Dye: Red 40
Color: Red


🔍 NEW FILE DETECTED: QEP075821__1__12-07-17-439.txt

TRANSMISSION SPECTRUM ANALYSIS
File: QEP075821__1__12-07-17-439.txt
Transmission Rise Point: 504.51 nm
Intensity at inflection: 3316.07
Rate of rise: 462.16
Identified Dye: Yellow 6
Color: Yellow-Orange


🔍 NEW FILE DETECTED: QEP075821__2__12-09-20-723.txt

TRANSMISSION SPECTRUM ANALYSIS
File: QEP075821__2__12-09-20-723.txt
Transmission Rise Point: 539.73 nm
Intensity at inflection: 9556.92
Rate of rise: 528.11
Identified Dye: Yellow 5
Color: Yellow


🔍 NEW FILE DETECTED: QEP075821__3__12-10-43-442.txt

TRANSMISSION SPECTRUM ANALYSIS
File: QEP075821__3__12-10-43-442.txt
Transmission Rise Point: 445.99 nm
Intensity at inflection: 2934.18
Rate of rise: 258.77
Identified Dye: Blue 1
Color: Blu

In [ ]:
m.park_tool()

In [ ]:
well = "A1"
m.move_to(labware2[well].x, labware2[well].y, s=1200)

In [ ]:
from spectrum_analyzer import test_file
test_file(r"C:\Users\Blab2\Desktop\Project\oceanview_data\QEP075821__0__17-46-34-323.txt")

In [ ]:
pump.set_valve_position("I")  # set valve to initial position

In [ ]:
pump.go_to_max_volume()     

In [ ]:
pump.set_valve_position("O")  # set valve to initial position

In [ ]:
pump.go_to_volume(0.03125)

In [ ]:
pump.go_to_volume(0.125)

In [ ]:
pump.go_to_volume(0)

In [ ]:
tip_manager.pickup_next_tip()

In [ ]:
m.remove_tip()

In [ ]:
tip_manager.reset_tip_counter()

In [ ]:
for i in range(5):

    tip_manager.pickup_next_tip()
    m.remove_tip()
    m.move_to(x=150, y=150)

In [ ]:
# Prime the water line from the input side, then push water to near the end of the output side then connect pipette

In [ ]:
# Syringe priming loop
# Adjust the range value to change number of priming cycles
for i in range(4):  # This will prime N times
    print(f"Priming cycle {i+1}")
    
    pump.set_valve_position("I")  # set valve to input position     
    pump.go_to_max_volume()       # draw fluid in
    
    pump.set_valve_position("O")  # set valve to output position
    pump.go_to_volume(0)          # expel fluid
    
    print(f"Cycle {i+1} complete")

print("Priming complete!")

In [ ]:
# Example: Clean probe between readings

# After taking a reading with the probe
m.move_to(z=100, s=1000)

# Arc method (what we're debugging)
m.clean_probe_arc(
    center_x=labware4["A1"].x,
    center_y=labware4["A1"].y,
    z_depth=40,
    radius=4,
    num_circles=10,
    speed=4000,
 )

In [ ]:
m.park_tool()

In [ ]:
m.move_to(x=150, y=150)

In [ ]:

##############################   READING FOR UNMIXED COLORS    ##############################

m.pickup_tool("dummy_probe")
m.move_to(labware2["A1"].x, labware2["A1"].y, s=1000)
m.move_to(z=-10, s=1000)
input("Press Enter to continue...")
m.move_to(z=75, s=1000)
m.move_to(labware2["B1"].x, labware2["B1"].y, s=1000)
m.move_to(z=-10, s=1000)
input("Press Enter to continue...")
m.move_to(z=75, s=1000)
m.move_to(labware2["C1"].x, labware2["C1"].y, s=1000)
m.move_to(z=-10, s=1000)
input("Press Enter to continue...")

m.move_to(z=100, s=1000)
m.park_tool()

##############################   COLOR MIXING    ##############################

m.pickup_tool("tube")


# Red
m.move_to(labware2["A1"].x, labware2["A1"].y, s=1000)
m.move_to(z=8, s=600)
pump.go_to_max_volume(wait=True)                                     
m.move_to(z=100, s=1200)


m.move_to(labware2["A4"].x, labware2["A4"].y, s=1000)
m.move_to(z=8, s=600)
pump.go_to_volume(0.125, wait=True)  
m.move_to(z=75, s=1000)
m.move_to(labware2["B4"].x, labware2["B4"].y, s=1000)
m.move_to(z=8, s=600)
pump.go_to_volume(0, wait=True)
m.move_to(z=100, s=1200)

# Yellow
m.move_to(labware2["B1"].x, labware2["B1"].y, s=1000)
m.move_to(z=8, s=600)
pump.go_to_max_volume(wait=True)                                                                    
m.move_to(z=100, s=1200)


m.move_to(labware2["B4"].x, labware2["B4"].y, s=1000)
m.move_to(z=8, s=600)
pump.go_to_volume(0.125, wait=True)  
m.move_to(z=75, s=1000)
m.move_to(labware2["C4"].x, labware2["C4"].y, s=1000)
m.move_to(z=8, s=600)
pump.go_to_volume(0, wait=True)
m.move_to(z=100, s=1200)

# Blue
m.move_to(labware2["C1"].x, labware2["C1"].y, s=1000)
m.move_to(z=8, s=600)
pump.go_to_max_volume(wait=True)                                                                            
m.move_to(z=100, s=1200)


m.move_to(labware2["A4"].x, labware2["A4"].y, s=1000)
m.move_to(z=8, s=600)
pump.go_to_volume(0.125, wait=True)  
m.move_to(z=75, s=1000)
m.move_to(labware2["C4"].x, labware2["C4"].y, s=1000)
m.move_to(z=8, s=600)
pump.go_to_volume(0, wait=True)

##############################   READING FOR MIXED COLORS    ##############################

m.pickup_tool("dummy_probe")
m.move_to(labware2["A4"].x, labware2["A4"].y, s=1000)
m.move_to(z=-10, s=1000)
input("Press Enter to continue...")
m.move_to(z=75, s=1000)
m.move_to(labware2["B4"].x, labware2["B4"].y, s=1000)
m.move_to(z=-10, s=1000)
input("Press Enter to continue...")
m.move_to(z=75, s=1000)
m.move_to(labware2["C4"].x, labware2["C4"].y, s=1000)
m.move_to(z=-10, s=1000)
input("Press Enter to continue...")

m.move_to(z=100, s=1000)
m.park_tool()


In [ ]:

##############################   READING FOR UNMIXED COLORS    ##############################

m.pickup_tool("dummy_probe")

m.move_to(labware3["A1"].x, labware3["A1"].y, s=2000)
m.move_to(z=0, s=2000)
input("Press Enter to continue once stored the refrence spectrum...")
m.move_to(z=75, s=2000)
input("Press Enter to continue once cleaned the probe...")

m.move_to(labware3["B2"].x, labware3["B2"].y, s=2000)
m.move_to(z=0, s=2000)
input("Turn the lights and lamp off and press Enter to continue once stored the blank beaker spectrum...")
m.move_to(z=75, s=2000)
input("Turn the lights and lamp on and press Enter to continue...")


#Red
m.move_to(labware2["A1"].x, labware2["A1"].y, s=2000)
m.move_to(z=-10, s=2000)
input("Press Enter to continue after storing the data...")


m.move_to(z=75, s=2000)
m.move_to(labware3["A1"].x, labware3["A1"].y, s=2000)
m.move_to(z=0, s=2000)
input("Press Enter to continue once stored the refrence spectrum...")
m.move_to(z=50, s=2000)
input("Press Enter to continue once cleaned the probe...")

m.move_to(labware3["B2"].x, labware3["B2"].y, s=2000)
m.move_to(z=0, s=2000)
input("Turn the lights and lamp off and press Enter to continue once stored the blank beaker spectrum...")
m.move_to(z=50, s=2000)
input("Turn the lights and lamp on and press Enter to continue...")


#Yellow
m.move_to(z=75, s=2000)
m.move_to(labware2["B1"].x, labware2["B1"].y, s=1000)
m.move_to(z=-10, s=2000)
input("Press Enter to continue after storing the data...")

m.move_to(z=75, s=2000)
m.move_to(labware3["A1"].x, labware3["A1"].y, s=2000)
m.move_to(z=0, s=2000)
input("Press Enter to continue once stored the refrence spectrum...")
m.move_to(z=50, s=2000)
input("Press Enter to continue once cleaned the probe...")

m.move_to(labware3["B2"].x, labware3["B2"].y, s=2000)
m.move_to(z=0, s=2000)
input("Turn the lights and lamp off and press Enter to continue once stored the blank beaker spectrum...")
m.move_to(z=50, s=2000)
input("Turn the lights and lamp on and press Enter to continue...")


#Blue
m.move_to(z=75, s=2000)
m.move_to(labware2["C1"].x, labware2["C1"].y, s=2000)
m.move_to(z=-10, s=2000)
input("Press Enter to continue after storing the data...")

m.move_to(z=75, s=2000)
m.move_to(labware3["A1"].x, labware3["A1"].y, s=2000)
m.move_to(z=0, s=2000)
input("Press Enter to continue once stored the refrence spectrum...")
m.move_to(z=50, s=2000)
input("Press Enter to continue once cleaned the probe...")

m.move_to(labware3["B2"].x, labware3["B2"].y, s=2000)
m.move_to(z=0, s=2000)
input("Turn the lights and lamp off and press Enter to continue once stored the blank beaker spectrum...")
m.move_to(z=50, s=2000)
input("Turn the lights and lamp on and press Enter to continue...")


m.move_to(z=100, s=2000)

##############################   COLOR MIXING    ##############################

m.pickup_tool("tube")

# Red - 8 pickups from A1, alternating between A4 and B4 (4 each)
for i in range(8):
    print(f"Red cycle {i+1}/8")
    
    # Pick up from A1
    m.move_to(labware2["A1"].x, labware2["A1"].y, s=2000)
    m.move_to(z=8, s=600)
    pump.go_to_max_volume(wait=True)                                     
    m.move_to(z=100, s=2000)
    
    # Dispense: first 4 cycles go to A4, next 4 go to B4
    if i < 4:
        # Dispense to A4
        m.move_to(labware2["A4"].x, labware2["A4"].y, s=2000)
        m.move_to(z=8, s=600)
        pump.go_to_volume(0, wait=True)
        m.move_to(z=100, s=2000)
    else:
        # Dispense to B4
        m.move_to(labware2["B4"].x, labware2["B4"].y, s=2000)
        m.move_to(z=8, s=600)
        pump.go_to_volume(0, wait=True)
        m.move_to(z=100, s=2000)

print("Red mixing complete!")

# Yellow - 8 pickups from B1, alternating between B4 and C4 (4 each)
for i in range(8):
    print(f"Yellow cycle {i+1}/8")
    
    # Pick up from B1
    m.move_to(labware2["B1"].x, labware2["B1"].y, s=2000)
    m.move_to(z=8, s=600)
    pump.go_to_max_volume(wait=True)                                                                    
    m.move_to(z=100, s=2000)
    
    # Dispense: first 4 cycles go to B4, next 4 go to C4
    if i < 4:
        # Dispense to B4
        m.move_to(labware2["B4"].x, labware2["B4"].y, s=2000)
        m.move_to(z=8, s=600)
        pump.go_to_volume(0, wait=True)
        m.move_to(z=100, s=2000)
    else:
        # Dispense to C4
        m.move_to(labware2["C4"].x, labware2["C4"].y, s=2000)
        m.move_to(z=8, s=600)
        pump.go_to_volume(0, wait=True)
        m.move_to(z=100, s=2000)

print("Yellow mixing complete!")

# Blue - 8 pickups from C1, alternating between A4 and C4 (4 each)
for i in range(8):
    print(f"Blue cycle {i+1}/8")
    
    # Pick up from C1
    m.move_to(labware2["C1"].x, labware2["C1"].y, s=2000)
    m.move_to(z=8, s=600)
    pump.go_to_max_volume(wait=True)                                                                            
    m.move_to(z=100, s=2000)
    
    # Dispense: first 4 cycles go to A4, next 4 go to C4
    if i < 4:
        # Dispense to A4
        m.move_to(labware2["A4"].x, labware2["A4"].y, s=2000)
        m.move_to(z=8, s=600)
        pump.go_to_volume(0, wait=True)
        m.move_to(z=100, s=2000)
    else:
        # Dispense to C4
        m.move_to(labware2["C4"].x, labware2["C4"].y, s=2000)
        m.move_to(z=8, s=600)
        pump.go_to_volume(0, wait=True)
        m.move_to(z=100, s=2000)

print("Blue mixing complete!")
print("All color mixing complete!")

##############################   READING FOR MIXED COLORS    ##############################

m.pickup_tool("dummy_probe")

m.move_to(labware3["A1"].x, labware3["A1"].y, s=2000)
m.move_to(z=0, s=2000)
input("Press Enter to continue once stored the refrence spectrum...")
m.move_to(z=75, s=2000)
input("Press Enter to continue once cleaned the probe...")

m.move_to(labware3["B2"].x, labware3["B2"].y, s=2000)
m.move_to(z=0, s=2000)
input("Turn the lights and lamp off and press Enter to continue once stored the blank beaker spectrum...")
m.move_to(z=75, s=2000)
input("Turn the lights and lamp on and press Enter to continue...")


#Red + Blue 
m.move_to(labware2["A1"].x, labware2["A1"].y, s=2000)
m.move_to(z=-10, s=2000)
input("Press Enter to continue after storing the data...")


m.move_to(z=75, s=2000)
m.move_to(labware3["A1"].x, labware3["A1"].y, s=2000)
m.move_to(z=0, s=2000)
input("Press Enter to continue once stored the refrence spectrum...")
m.move_to(z=50, s=2000)
input("Press Enter to continue once cleaned the probe...")

m.move_to(labware3["B2"].x, labware3["B2"].y, s=2000)
m.move_to(z=0, s=2000)
input("Turn the lights and lamp off and press Enter to continue once stored the blank beaker spectrum...")
m.move_to(z=50, s=2000)
input("Turn the lights and lamp on and press Enter to continue...")


#Yellow + Red
m.move_to(z=75, s=2000)
m.move_to(labware2["B1"].x, labware2["B1"].y, s=1000)
m.move_to(z=-10, s=2000)
input("Press Enter to continue...")

m.move_to(z=75, s=2000)
m.move_to(labware3["A1"].x, labware3["A1"].y, s=2000)
m.move_to(z=0, s=2000)
input("Press Enter to continue once stored the refrence spectrum...")
m.move_to(z=50, s=2000)
input("Press Enter to continue once cleaned the probe...")

m.move_to(labware3["B2"].x, labware3["B2"].y, s=2000)
m.move_to(z=0, s=2000)
input("Turn the lights and lamp off and press Enter to continue once stored the blank beaker spectrum...")
m.move_to(z=50, s=2000)
input("Turn the lights and lamp on and press Enter to continue...")


#Blue + Yellow
m.move_to(z=75, s=2000)
m.move_to(labware2["C1"].x, labware2["C1"].y, s=2000)
m.move_to(z=-10, s=2000)
input("Press Enter to continue...")

m.move_to(z=75, s=2000)
m.move_to(labware3["A1"].x, labware3["A1"].y, s=2000)
m.move_to(z=0, s=2000)
input("Press Enter to continue once stored the refrence spectrum...")
m.move_to(z=50, s=2000)
input("Press Enter to continue once cleaned the probe...")

m.move_to(labware3["B2"].x, labware3["B2"].y, s=2000)
m.move_to(z=0, s=2000)
input("Turn the lights and lamp off and press Enter to continue once stored the blank beaker spectrum...")
m.move_to(z=50, s=2000)
input("Turn the lights and lamp on and press Enter to continue...")



m.move_to(z=100, s=2000)
m.park_tool()


In [ ]:
m.pickup_tool("dummy_probe")

m.move_to(labware3["A1"].x, labware3["A1"].y, s=2000)


In [ ]:
m.move_to(z=100, s=2000)

In [ ]:
m.move_to(labware2["B1"].x, labware2["B1"].y, s=2000)


In [ ]:

##############################   READING FOR UNMIXED COLORS    ##############################

m.pickup_tool("dummy_probe")


#Red
m.move_to(labware2["A1"].x, labware2["A1"].y, s=2000)
m.move_to(z=-10, s=2000)
input("Press Enter to continue after storing the data...")


m.move_to(z=75, s=2000)
m.move_to(labware3["A1"].x, labware3["A1"].y, s=2000)
m.move_to(z=0, s=2000)
m.move_to(z=50, s=2000)
input("Press Enter to continue once cleaned the probe...")


#Yellow
m.move_to(z=75, s=2000)
m.move_to(labware2["B1"].x, labware2["B1"].y, s=1000)
m.move_to(z=-10, s=2000)
input("Press Enter to continue after storing the data...")

m.move_to(z=75, s=2000)
m.move_to(labware3["A1"].x, labware3["A1"].y, s=2000)
m.move_to(z=0, s=2000)
m.move_to(z=50, s=2000)
input("Press Enter to continue once cleaned the probe...")


#Blue
m.move_to(z=75, s=2000)
m.move_to(labware2["C1"].x, labware2["C1"].y, s=2000)
m.move_to(z=-10, s=2000)
input("Press Enter to continue after storing the data...")

m.move_to(z=75, s=2000)
m.move_to(labware3["A1"].x, labware3["A1"].y, s=2000)
m.move_to(z=0, s=2000)
m.move_to(z=50, s=2000)
input("Press Enter to continue once cleaned the probe...")


m.move_to(z=100, s=2000)

##############################   COLOR MIXING    ##############################

m.pickup_tool("tube")

# Red - 8 pickups from A1, alternating between A4 and B4 (4 each)
for i in range(8):
    print(f"Red cycle {i+1}/8")
    
    # Pick up from A1
    m.move_to(labware2["A1"].x, labware2["A1"].y, s=2000)
    m.move_to(z=8, s=600)
    pump.go_to_max_volume(wait=True)                                     
    m.move_to(z=100, s=2000)
    
    # Dispense: first 4 cycles go to A4, next 4 go to B4
    if i < 4:
        # Dispense to A4
        m.move_to(labware2["A4"].x, labware2["A4"].y, s=2000)
        m.move_to(z=8, s=600)
        pump.go_to_volume(0, wait=True)
        m.move_to(z=100, s=2000)
    else:
        # Dispense to B4
        m.move_to(labware2["B4"].x, labware2["B4"].y, s=2000)
        m.move_to(z=8, s=600)
        pump.go_to_volume(0, wait=True)
        m.move_to(z=100, s=2000)

print("Red mixing complete!")

m.move_to(labware3["A2"].x, labware3["A2"].y, s=2000)
m.move_to(z=20, s=2000)
pump.go_to_max_volume(wait=True)
pump.go_to_volume(0, wait=True)
m.move_to(z=100, s=2000)


# Yellow - 8 pickups from B1, alternating between B4 and C4 (4 each)
for i in range(8):
    print(f"Yellow cycle {i+1}/8")
    
    # Pick up from B1
    m.move_to(labware2["B1"].x, labware2["B1"].y, s=2000)
    m.move_to(z=8, s=600)
    pump.go_to_max_volume(wait=True)                                                                    
    m.move_to(z=100, s=2000)
    
    # Dispense: first 4 cycles go to B4, next 4 go to C4
    if i < 4:
        # Dispense to B4
        m.move_to(labware2["B4"].x, labware2["B4"].y, s=2000)
        m.move_to(z=8, s=600)
        pump.go_to_volume(0, wait=True)
        m.move_to(z=100, s=2000)
    else:
        # Dispense to C4
        m.move_to(labware2["C4"].x, labware2["C4"].y, s=2000)
        m.move_to(z=8, s=600)
        pump.go_to_volume(0, wait=True)
        m.move_to(z=100, s=2000)

print("Yellow mixing complete!")


m.move_to(labware3["A2"].x, labware3["A2"].y, s=2000)
m.move_to(z=20, s=2000)
pump.go_to_max_volume(wait=True)
pump.go_to_volume(0, wait=True)
m.move_to(z=100, s=2000)


# Blue - 8 pickups from C1, alternating between A4 and C4 (4 each)
for i in range(8):
    print(f"Blue cycle {i+1}/8")
    
    # Pick up from C1
    m.move_to(labware2["C1"].x, labware2["C1"].y, s=2000)
    m.move_to(z=8, s=600)
    pump.go_to_max_volume(wait=True)                                                                            
    m.move_to(z=100, s=2000)
    
    # Dispense: first 4 cycles go to A4, next 4 go to C4
    if i < 4:
        # Dispense to A4
        m.move_to(labware2["A4"].x, labware2["A4"].y, s=2000)
        m.move_to(z=8, s=600)
        pump.go_to_volume(0, wait=True)
        m.move_to(z=100, s=2000)
    else:
        # Dispense to C4
        m.move_to(labware2["C4"].x, labware2["C4"].y, s=2000)
        m.move_to(z=8, s=600)
        pump.go_to_volume(0, wait=True)
        m.move_to(z=100, s=2000)

print("Blue mixing complete!")
print("All color mixing complete!")


m.move_to(labware3["A2"].x, labware3["A2"].y, s=2000)
m.move_to(z=20, s=2000)
pump.go_to_max_volume(wait=True)
pump.go_to_volume(0, wait=True)
m.move_to(z=100, s=2000)


##############################   READING FOR MIXED COLORS    ##############################

m.pickup_tool("dummy_probe")

m.move_to(labware3["A1"].x, labware3["A1"].y, s=2000)
m.move_to(z=0, s=2000)
m.move_to(z=75, s=2000)
input("Press Enter to continue once cleaned the probe...")


#Red + Blue 
m.move_to(labware2["A4"].x, labware2["A4"].y, s=2000)
m.move_to(z=-10, s=2000)
input("Press Enter to continue after storing the data...")


m.move_to(z=75, s=2000)
m.move_to(labware3["A1"].x, labware3["A1"].y, s=2000)
m.move_to(z=0, s=2000)
m.move_to(z=50, s=2000)
input("Press Enter to continue once cleaned the probe...")


#Yellow + Red
m.move_to(z=75, s=2000)
m.move_to(labware2["B4"].x, labware2["B4"].y, s=1000)
m.move_to(z=-10, s=2000)
input("Press Enter to continue after storing the data...")

m.move_to(z=75, s=2000)
m.move_to(labware3["A1"].x, labware3["A1"].y, s=2000)
m.move_to(z=0, s=2000)
m.move_to(z=50, s=2000)
input("Press Enter to continue once cleaned the probe...")


#Blue + Yellow

m.move_to(z=75, s=2000)
m.move_to(labware2["C4"].x, labware2["C4"].y, s=2000)
m.move_to(z=-10, s=2000)
input("Press Enter to continue after storing the data...")

m.move_to(z=75, s=2000)
m.move_to(labware3["A1"].x, labware3["A1"].y, s=2000)
m.move_to(z=0, s=2000)
m.move_to(z=50, s=2000)
input("Press Enter to continue once cleaned the probe...")



m.move_to(z=100, s=2000)
m.park_tool()


In [18]:


# G1 - команда движения (F1000 - скорость 1000 мм/мин)
m.gcode("G1 Z50 F1000")

'Error: G1: insufficient axes homed'